# Synthetic MuSiQue: 30-question review batch

Target **five structurally valid, normalized-exact-unique questions for each of the six hop subtypes**.
This is a diagnostic sample; accepted records still need semantic review.

Select the **capstone** Python kernel. Run setup and definitions, then deliberately enable the generation cell when ready.
**Generation makes paid calls. It is disabled by default.** No calls are needed to reopen saved outputs or review them.

The default cap is 10 requests per subtype, with at most 2 model calls per request: **120 client calls maximum**,
usually 30 calls if every first attempt succeeds. SDK-internal retries are separate. The batch may finish below 30 accepted
records if a subtype reaches its cap. API/programming errors stop the batch and retain available history.


In [7]:
import csv
import hashlib
import html
import json
import random
import sys
from collections import Counter
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from IPython.display import HTML, display

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src/mas_sae/data/synthetic").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository.")
sys.path.insert(0, str(ROOT / "src"))

from mas_sae.data.synthetic import DuplicateIndex, validate_candidate
from mas_sae.data.synthetic.generation import PROMPT_VERSION, ParagraphPool, generate_one
from mas_sae.data.synthetic.generation_cli import DEFAULT_MODEL_ID, make_client
from mas_sae.data.synthetic.hop_types import HOP_TYPES

MODEL_ID = DEFAULT_MODEL_ID
HOP_TYPES_TO_REVIEW = tuple(HOP_TYPES)
TARGET_PER_TYPE = 5
MAX_REQUESTS_PER_TYPE = 10
MAX_ATTEMPTS = 2
MAX_TOKENS = 6000
TEMPERATURE = None  # Omit temperature for Opus.
SEED = 42  # Local sampling/shuffling only; not a model-generation seed.
ORIGINAL_FILES = (
    ROOT / "data/MuSiQue/clean/train.json",
    ROOT / "data/MuSiQue/clean/validation.json",
)
RUN_DIR = ROOT / "data/MuSiQue/synthetic/runs" / (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_review30_" + uuid4().hex[:8]
)
print(f"Model: {MODEL_ID} | prompt: {PROMPT_VERSION}")
print(f"Target: {TARGET_PER_TYPE * len(HOP_TYPES_TO_REVIEW)} accepted records")
print(f"Maximum client calls: {MAX_REQUESTS_PER_TYPE * len(HOP_TYPES_TO_REVIEW) * MAX_ATTEMPTS}")
print(f"Output: {RUN_DIR}")


Model: us.anthropic.claude-opus-5 | prompt: v3
Target: 30 accepted records
Maximum client calls: 120
Output: /home/ubuntu/capstone/fall-2026-group3/data/MuSiQue/synthetic/runs/20261030T163044Z_review30_3ee751b1


## Batch definitions — no API calls

This function reuses `generate_one`, `validate_candidate`, and `DuplicateIndex`.
It cycles through subtypes, replacing structural failures or exact duplicates until each quota is met or its request cap is reached.
The duplicate index covers both original splits and this batch, **not other synthetic runs**.

Artifacts: `candidates.jsonl` (all generated candidates), `accepted.jsonl` (structurally valid and unique),
`validation.jsonl` (per-request results), `provenance.jsonl` (raw responses, repairs, errors, token usage), and `summary.json`.
An existing output directory is refused so re-executing generation cannot overwrite a prior batch.


In [8]:
def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def run_review_batch(output_dir, *, client=None):
    output_dir = Path(output_dir)
    if not 0 < TARGET_PER_TYPE <= MAX_REQUESTS_PER_TYPE:
        raise ValueError("Require 0 < TARGET_PER_TYPE <= MAX_REQUESTS_PER_TYPE.")
    for path in ORIGINAL_FILES:
        if not path.is_file():
            raise FileNotFoundError(path)
    pool = ParagraphPool.from_jsonl(ORIGINAL_FILES[0])
    output_dir.mkdir(parents=True, exist_ok=False)
    rng = random.Random(SEED)
    requests, accepted = Counter(), Counter()
    totals = Counter(api_calls=0, completed_responses=0, input_tokens=0, output_tokens=0)
    status, fatal_error = "running", None
    owns_client = client is None

    def write_line(stream, value):
        stream.write(json.dumps(value, ensure_ascii=False) + "\n")
        stream.flush()

    def summary():
        return {
            "status": status, "fatal_error": fatal_error,
            "model_id": MODEL_ID if client is None else client.model_id,
            "prompt_version": PROMPT_VERSION, "temperature": TEMPERATURE,
            "max_tokens": MAX_TOKENS, "max_attempts": MAX_ATTEMPTS, "seed": SEED,
            "target_per_type": TARGET_PER_TYPE,
            "max_requests_per_type": MAX_REQUESTS_PER_TYPE,
            "requests_by_type": {hop: requests[hop] for hop in HOP_TYPES_TO_REVIEW},
            "accepted_by_type": {hop: accepted[hop] for hop in HOP_TYPES_TO_REVIEW},
            "accepted_total": sum(accepted.values()), **totals,
            "original_files": [str(path) for path in ORIGINAL_FILES],
            "output_dir": str(output_dir),
        }

    try:
        with (
            DuplicateIndex(output_dir / "question_index.sqlite") as index,
            (output_dir / "candidates.jsonl").open("w", encoding="utf-8") as candidates_file,
            (output_dir / "accepted.jsonl").open("w", encoding="utf-8") as accepted_file,
            (output_dir / "validation.jsonl").open("w", encoding="utf-8") as validation_file,
            (output_dir / "provenance.jsonl").open("w", encoding="utf-8") as provenance_file,
        ):
            for path in ORIGINAL_FILES:
                index.import_jsonl(path, source=str(path))
            if owns_client:
                client = make_client(MODEL_ID)
            for _ in range(MAX_REQUESTS_PER_TYPE):
                for hop in HOP_TYPES_TO_REVIEW:
                    if accepted[hop] >= TARGET_PER_TYPE:
                        continue
                    requests[hop] += 1
                    request_id = f"{hop}_request_{requests[hop]:03d}"
                    provenance = {"hop_type": hop, "attempts": []}
                    try:
                        candidate, provenance = generate_one(
                            client, hop, pool, rng, max_attempts=MAX_ATTEMPTS,
                            max_tokens=MAX_TOKENS, temperature=TEMPERATURE,
                            provenance=provenance,
                        )
                    except BaseException as exc:
                        provenance["error"] = f"{type(exc).__name__}: {exc}"
                        raise
                    finally:
                        provenance["request_id"] = request_id
                        totals["api_calls"] += provenance.get("api_calls", 0)
                        for attempt in provenance["attempts"]:
                            totals["completed_responses"] += 1
                            totals["input_tokens"] += attempt["input_tokens"]
                            totals["output_tokens"] += attempt["output_tokens"]
                        write_line(provenance_file, provenance)

                    result = {"request_id": request_id, "hop_type": hop,
                              "record_id": candidate["id"] if candidate else None,
                              "accepted": False, "errors": []}
                    if candidate is None:
                        result["errors"] = [{"code": "generation_exhausted",
                                             "message": "See this request's provenance attempts."}]
                    else:
                        write_line(candidates_file, candidate)
                        result["errors"] = [asdict(issue) for issue in validate_candidate(candidate).errors]
                        if not result["errors"]:
                            duplicate = index.register_question(
                                candidate["id"], candidate["question"], source=str(output_dir)
                            )
                            if duplicate:
                                result["errors"] = [{"code": "duplicate", **asdict(duplicate)}]
                            else:
                                write_line(accepted_file, candidate)
                                accepted[hop] += 1
                                result["accepted"] = True
                    write_line(validation_file, result)
                    print(f"{request_id}: {'accepted' if result['accepted'] else 'rejected'} "
                          f"({accepted[hop]}/{TARGET_PER_TYPE})", flush=True)
                if all(accepted[hop] >= TARGET_PER_TYPE for hop in HOP_TYPES_TO_REVIEW):
                    break
            status = "completed" if all(
                accepted[hop] >= TARGET_PER_TYPE for hop in HOP_TYPES_TO_REVIEW
            ) else "quota_not_met"
    except BaseException as exc:
        status = "interrupted" if isinstance(exc, KeyboardInterrupt) else "failed"
        fatal_error = f"{type(exc).__name__}: {exc}"
        raise
    finally:
        try:
            if owns_client and client is not None:
                client.close()
        finally:
            (output_dir / "summary.json").write_text(
                json.dumps(summary(), indent=2) + "\n", encoding="utf-8"
            )
    return summary()


## Generate — paid calls only when enabled

Change `RUN_GENERATION` to `True` and execute this cell deliberately. A new batch refuses an existing `RUN_DIR`.
To generate a separate batch, rerun configuration to obtain a new directory. Reset the switch afterward.
If a run fails or is interrupted, load its saved output below; this notebook does not automatically resume or regenerate it.


In [9]:
RUN_GENERATION = True
if RUN_GENERATION:
    batch_summary = run_review_batch(RUN_DIR)
    print(json.dumps(batch_summary, indent=2))
else:
    print("Generation disabled. No API calls made.")


2hop_request_001: accepted (1/5)
3hop1_request_001: accepted (1/5)
3hop2_request_001: accepted (1/5)
4hop1_request_001: accepted (1/5)
4hop2_request_001: accepted (1/5)
4hop3_request_001: accepted (1/5)
2hop_request_002: accepted (2/5)
3hop1_request_002: accepted (2/5)
3hop2_request_002: accepted (2/5)
4hop1_request_002: accepted (2/5)
4hop2_request_002: accepted (2/5)
4hop3_request_002: accepted (2/5)
2hop_request_003: accepted (3/5)
3hop1_request_003: accepted (3/5)
3hop2_request_003: accepted (3/5)
4hop1_request_003: accepted (3/5)
4hop2_request_003: accepted (3/5)
4hop3_request_003: accepted (3/5)
2hop_request_004: accepted (4/5)
3hop1_request_004: accepted (4/5)
3hop2_request_004: accepted (4/5)
4hop1_request_004: accepted (4/5)
4hop2_request_004: accepted (4/5)
4hop3_request_004: accepted (4/5)
2hop_request_005: accepted (5/5)
3hop1_request_005: accepted (5/5)
3hop2_request_005: accepted (5/5)
4hop1_request_005: accepted (5/5)
4hop2_request_005: accepted (5/5)
4hop3_request_005: 

## Load saved outputs — no API calls

For a previous session, replace `REVIEW_RUN_DIR` with its saved directory path.
You can load a partial run after an error. These cells never call generation.


In [10]:
REVIEW_RUN_DIR = RUN_DIR  # Or: ROOT / "data/MuSiQue/synthetic/runs/<saved-run>"

if not (REVIEW_RUN_DIR / "summary.json").is_file():
    print("No saved batch here yet. Generate it explicitly or set REVIEW_RUN_DIR to an existing run.")
    records, provenance_rows, validation_rows = [], [], []
else:
    saved_summary = json.loads((REVIEW_RUN_DIR / "summary.json").read_text())
    records = read_jsonl(REVIEW_RUN_DIR / "accepted.jsonl") if (REVIEW_RUN_DIR / "accepted.jsonl").exists() else []
    provenance_rows = read_jsonl(REVIEW_RUN_DIR / "provenance.jsonl") if (REVIEW_RUN_DIR / "provenance.jsonl").exists() else []
    validation_rows = read_jsonl(REVIEW_RUN_DIR / "validation.jsonl") if (REVIEW_RUN_DIR / "validation.jsonl").exists() else []
    print(json.dumps(saved_summary, indent=2))
    rows = []
    for hop in HOP_TYPES_TO_REVIEW:
        requested = saved_summary["requests_by_type"].get(hop, 0)
        count = saved_summary["accepted_by_type"].get(hop, 0)
        rate = f"{count / requested:.0%}" if requested else "—"
        rows.append(f"<tr><td>{hop}</td><td>{requested}</td><td>{count}</td><td>{rate}</td></tr>")
    display(HTML("<table><tr><th>Subtype</th><th>Requests</th><th>Accepted</th>"
                 "<th>Acceptance per request</th></tr>" + "".join(rows) + "</table>"))
    failure_counts = Counter(
        error["code"] for row in validation_rows for error in row["errors"]
    )
    print("Request rejection reasons:", dict(failure_counts))
    print("A successful request can include an earlier rejected model response; inspect provenance for those details.")


{
  "status": "completed",
  "fatal_error": null,
  "model_id": "us.anthropic.claude-opus-5",
  "prompt_version": "v3",
  "temperature": null,
  "max_tokens": 6000,
  "max_attempts": 2,
  "seed": 42,
  "target_per_type": 5,
  "max_requests_per_type": 10,
  "requests_by_type": {
    "2hop": 5,
    "3hop1": 5,
    "3hop2": 5,
    "4hop1": 5,
    "4hop2": 5,
    "4hop3": 5
  },
  "accepted_by_type": {
    "2hop": 5,
    "3hop1": 5,
    "3hop2": 5,
    "4hop1": 5,
    "4hop2": 5,
    "4hop3": 5
  },
  "accepted_total": 30,
  "api_calls": 31,
  "completed_responses": 31,
  "input_tokens": 43103,
  "output_tokens": 66248,
  "original_files": [
    "/home/ubuntu/capstone/fall-2026-group3/data/MuSiQue/clean/train.json",
    "/home/ubuntu/capstone/fall-2026-group3/data/MuSiQue/clean/validation.json"
  ],
  "output_dir": "/home/ubuntu/capstone/fall-2026-group3/data/MuSiQue/synthetic/runs/20261030T163044Z_review30_3ee751b1"
}


Subtype,Requests,Accepted,Acceptance per request
2hop,5,5,100%
3hop1,5,5,100%
3hop2,5,5,100%
4hop1,5,5,100%
4hop2,5,5,100%
4hop3,5,5,100%


Request rejection reasons: {}
A successful request can include an earlier rejected model response; inspect provenance for those details.


## Inspect questions and documents

Change `RECORD_INDEX` to select a question (0–29 for a full batch). Inspect all eight criteria below.
Supporting paragraphs and step references use the saved shuffled indices. Distractors and generation history can be expanded.
Passing structural checks does not prove necessary hops or factual consistency.


In [11]:
def show_record(record_index):
    if not records:
        print("Load a saved batch first.")
        return
    record = records[record_index]
    escape = lambda value: html.escape(str(value))
    parts = [f"<h3>{record_index + 1}/{len(records)} — {escape(record['id'])}</h3>",
             f"<p><b>Question:</b> {escape(record['question'])}</p>",
             f"<p><b>Answer:</b> {escape(record['answer'])} "
             f"<b>Aliases:</b> {escape(record['answer_aliases'])}</p>", "<h4>Decomposition</h4><ol>"]
    for step in record["question_decomposition"]:
        parts.append(f"<li>{escape(step['question'])} → <b>{escape(step['answer'])}</b> "
                     f"(paragraph {step['paragraph_support_idx']})</li>")
    parts.append("</ol><h4>Supporting paragraphs</h4>")
    for paragraph in record["paragraphs"]:
        if paragraph["is_supporting"]:
            parts.append(f"<p><b>[{paragraph['idx']}] {escape(paragraph['title'])}</b><br>"
                         f"{escape(paragraph['paragraph_text'])}</p>")
    parts.append("<details><summary>Distractors</summary>")
    for paragraph in record["paragraphs"]:
        if not paragraph["is_supporting"]:
            parts.append(f"<p><b>[{paragraph['idx']}] {escape(paragraph['title'])}</b><br>"
                         f"{escape(paragraph['paragraph_text'])}</p>")
    parts.append("</details>")
    history = [row for row in provenance_rows if row.get("record_id") == record["id"]]
    parts.append("<details><summary>Generation history and usage</summary><pre>"
                 + escape(json.dumps(history, ensure_ascii=False, indent=2)) + "</pre></details>")
    display(HTML("".join(parts)))


RECORD_INDEX = 0
show_record(RECORD_INDEX)


## Record manual review decisions

Create a CSV template below, then edit it in a spreadsheet or editor. Existing reviews are never overwritten.
Each row includes the record ID and a hash of its exact contents. This is a review worksheet, **not a release exporter**;
`accepted.jsonl` remains the automated-validation output and is not automatically approved for collection.

For each criterion use `pass`, `fail`, or `uncertain`:

| Criterion | Review question |
|---|---|
| answer_support | Do the documents establish one unambiguous answer? |
| internal_consistency | Are the documents and answers mutually consistent? |
| necessary_hops | What necessary information disappears if each supporting paragraph is removed? |
| single_hop_steps | Does each step express one direct lookup? |
| dependency_shape | Are the referenced earlier answers actually needed? |
| no_shortcuts | Can the intended reasoning be bypassed? |
| distractor_quality | Are distractors plausible without contradictions, shortcuts, or alternative answers? |
| question_clarity | Is the main question understandable and sufficiently specific? |

Use `approved`, `rejected`, or `pending` for `decision`; leave uncertain examples pending.
Add your reviewer name and a brief reason. Adoption/retention outcomes are evaluated separately after collection.


In [12]:
CRITERIA = (
    "answer_support", "internal_consistency", "necessary_hops", "single_hop_steps",
    "dependency_shape", "no_shortcuts", "distractor_quality", "question_clarity",
)
review_path = REVIEW_RUN_DIR / "manual_review.csv"
if not records:
    print("No accepted records loaded; no review file written.")
elif review_path.exists():
    print(f"Keeping existing reviews: {review_path}")
else:
    fields = ["record_id", "candidate_sha256", "hop_type", "decision", "reviewer", *CRITERIA, "notes"]
    with review_path.open("x", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        for record in records:
            canonical = json.dumps(record, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
            writer.writerow({
                "record_id": record["id"],
                "candidate_sha256": hashlib.sha256(canonical.encode("utf-8")).hexdigest(),
                "hop_type": record["id"].split("__", 1)[0],
                "decision": "pending",
            })
    print(f"Review worksheet: {review_path}")


Review worksheet: /home/ubuntu/capstone/fall-2026-group3/data/MuSiQue/synthetic/runs/20261030T163044Z_review30_3ee751b1/manual_review.csv
